# Embed MCC + VSIC → artifact (.npz) on Google Colab

Notebook này chạy **bước `embed`**: nhúng toàn bộ MCC + VSIC bằng **Qwen3-Embedding** và rerank bằng **Qwen3-Reranker** in-process (sentence-transformers) trên GPU của Colab, rồi ghi ra file artifact `embed-artifact.npz` lên Google Drive.

Sau khi chạy xong, tải `embed-artifact.npz` về máy local (hoặc dùng trực tiếp trên Drive) rồi chạy `map-vsic-mcc` ở bước 2.

> **Lưu ý:** Không cần cài Ollama. Embedding và reranking chạy trực tiếp qua GPU bằng `sentence-transformers`.

## 1. Kết nối Google Drive
Artifact sẽ được ghi trực tiếp lên Google Drive để tránh mất dữ liệu khi runtime bị reset.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. Clone code và cài đặt Dependencies
Clone repository về Google Drive, sau đó cài các thư viện cần thiết (`colab/requirements-embed.txt`).

In [ ]:
import os

PROJECT_DIR = "/content/drive/MyDrive/projects/mcc-lens"
REPO_URL = "https://github.com/hatuan314/mcc-lens.git"

os.makedirs(os.path.dirname(PROJECT_DIR), exist_ok=True)

if os.path.isdir(os.path.join(PROJECT_DIR, ".git")):
    os.chdir(PROJECT_DIR)
    !git pull
else:
    !git clone {REPO_URL} {PROJECT_DIR}
    os.chdir(PROJECT_DIR)

!pip install -q -r colab/requirements-embed.txt

## 3. Chạy bước Embed
Dùng flag `--gdrive-output-dir` để ghi `embed-artifact.npz` vào thư mục trên Drive.

Đầu vào mặc định: `output/mcc-visa.json` + `output/vsic-vn.json` (đảm bảo chúng đã có trong repo/Drive).

In [ ]:
!python3 main.py embed \
  --mcc-input output/mcc-visa.json \
  --vsic-input output/vsic-vn.json \
  --gdrive-output-dir /content/drive/MyDrive/projects/mcc-lens \
  --embedding-model Qwen/Qwen3-Embedding \
  --reranker-model Qwen/Qwen3-Reranker \
  --cosine-top-k 100 \
  --rerank-top-n 20

## 4. Kiểm tra artifact
Xác nhận file `embed-artifact.npz` đã được ghi và xem nhanh shape + meta.

In [ ]:
import json
import numpy as np

ARTIFACT = "/content/drive/MyDrive/projects/mcc-lens/embed-artifact.npz"
data = np.load(ARTIFACT, allow_pickle=True)
print("mcc_vectors:", data["mcc_vectors"].shape)
print("vsic_vectors:", data["vsic_vectors"].shape)
print("reranked_mcc_indices:", data["reranked_mcc_indices"].shape)
print("rerank_scores:", data["rerank_scores"].shape)
print("meta:", json.loads(str(data["meta"])))